# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
visible = df[(df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20)].copy()

tier_median_ctr = visible.groupby('position_tier')['ctr'].transform('median')
visible['ctr_gap'] = tier_median_ctr - visible['ctr']
gap_threshold = visible['ctr_gap'].quantile(0.75)
visible['needs_review'] = (visible['ctr_gap'] >= gap_threshold).astype(int)

feature_cols = [
    'search_volume', 'competition', 'cpc', 'content_type', 'main_intent',
    'word_count', 'char_count', 'provider_used', 'model_used',
    'impressions_90d', 'days_with_impressions', 'content_age_days',
    'days_since_last_update', 'avg_position', 'position_tier',
    'impression_tier', 'freshness_tier', 'age_tier', 'word_count_tier', 'char_count_tier'
]
categorical_cols = ['content_type', 'main_intent', 'provider_used', 'model_used',
                     'position_tier', 'impression_tier', 'freshness_tier',
                     'age_tier', 'word_count_tier', 'char_count_tier']
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

X = visible[feature_cols].copy()
X[numeric_cols] = X[numeric_cols].fillna(0)
X[categorical_cols] = X[categorical_cols].fillna('unknown').astype(str)
y = visible['needs_review']

# same honest grouped split as ML-08/ML-09
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=visible['client_id']))

prep = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols),
])
model = Pipeline([('prep', prep), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
model.fit(X.iloc[train_idx], y.iloc[train_idx])

# score every visible page (this is what a reviewer would actually open)
visible['model_score'] = model.predict_proba(X)[:, 1]
print(f"scored {len(visible):,} pages")
print(f"reminder — honest test-set precision@50 from ML-09: 0.400 (this is what I trust for reporting)")

scored 12,023 pages
reminder — honest test-set precision@50 from ML-09: 0.400 (this is what I trust for reporting)


In [2]:
def build_reason(row):
    reasons = []
    if row['position_tier'] == 'top_3':
        reasons.append("ranks in the top 3 but still underperforms")
    elif row['position_tier'] == 'page_1':
        reasons.append("ranks on page 1")
    elif row['position_tier'] == 'striking':
        reasons.append("ranks just outside page 1 (striking distance)")

    if row['freshness_tier'] in ['91-180', '181+']:
        reasons.append(f"hasn't been updated in {row['days_since_last_update']} days")
    elif row['freshness_tier'] == '0-30':
        reasons.append("was updated recently, so a stale title likely isn't the issue")

    if row['search_volume'] and row['search_volume'] > 0:
        reasons.append(f"has real search demand ({int(row['search_volume'])}/mo)")
    else:
        reasons.append("has little measured search demand for its main keyword")

    return "; ".join(reasons)

queue = visible.sort_values('model_score', ascending=False).copy()
queue['reason'] = queue.apply(build_reason, axis=1)
queue['action'] = 'review_title_and_meta_description'
queue['reason_code'] = 'model_flagged_likely_underperformer'

out_cols = ['content_id', 'client_id', 'model_score', 'action', 'reason_code', 'reason',
            'position_tier', 'avg_position', 'freshness_tier', 'search_volume']
queue[out_cols].head(10)

,content_id,client_id,model_score,action,reason_code,reason,position_tier,avg_position,freshness_tier,search_volume
13502,content_f76ccf7a7834,client_19581e27de,0.998626,review_title_and_meta_description,model_flagged_likely_underperformer,"ranks on page 1; was updated recently, so a st...",page_1,9.5,0-30,49500.0
5287,content_8ca50876b0df,client_3fdba35f04,0.975935,review_title_and_meta_description,model_flagged_likely_underperformer,ranks just outside page 1 (striking distance);...,striking,18.3,0-30,40500.0
2553,content_eb1510f4b5f1,client_3fdba35f04,0.968137,review_title_and_meta_description,model_flagged_likely_underperformer,ranks just outside page 1 (striking distance);...,striking,14.7,31-90,33100.0
2074,content_6ef3dcb7be11,client_4e07408562,0.941618,review_title_and_meta_description,model_flagged_likely_underperformer,"ranks on page 1; was updated recently, so a st...",page_1,6.0,0-30,27100.0
23206,content_f854023b075b,client_19581e27de,0.926418,review_title_and_meta_description,model_flagged_likely_underperformer,ranks on page 1; hasn't been updated in 104 da...,page_1,5.1,91-180,18100.0
22507,content_aa6fbeaf8434,client_19581e27de,0.895884,review_title_and_meta_description,model_flagged_likely_underperformer,ranks just outside page 1 (striking distance);...,striking,17.0,0-30,18100.0
17859,content_5c5fab9d41e7,client_4e07408562,0.879116,review_title_and_meta_description,model_flagged_likely_underperformer,ranks on page 1; hasn't been updated in 104 da...,page_1,5.4,91-180,22200.0
693,content_3d68cd06523d,client_d029fa3a95,0.874332,review_title_and_meta_description,model_flagged_likely_underperformer,"ranks on page 1; was updated recently, so a st...",page_1,10.0,0-30,0.0
3752,content_e7eb94e121b9,client_6208ef0f77,0.868255,review_title_and_meta_description,model_flagged_likely_underperformer,ranks just outside page 1 (striking distance);...,striking,15.6,91-180,22200.0
16621,content_06531199cd40,client_d029fa3a95,0.864938,review_title_and_meta_description,model_flagged_likely_underperformer,"ranks on page 1; was updated recently, so a st...",page_1,8.0,0-30,0.0


**What this queue is:** Instead of just a bare score, each page gets a reason built from its actual features (position tier, freshness, search demand), something a reviewer can read and trust in a few seconds, not just a number they have to interpret.

**What I notice in my own top 10:** Several of the highest scored pages were "updated recently" according to the reason text which means a stale title probably isn't the actual problem for them, since they've already been touched. That echoes the exact pattern I found in ML-08's error analysis that is the model leans hard on freshness, and a recent update doesn't guarantee good CTR. Two rows also have search_volume = 0, meaning the tracked keyword has little measured demand, worth a reviewer double-checking real traffic sources before assuming that a title fix will help.

**Reminder on trust:** The score ranking these pages is the same honest, grouped-split model from ML-08/ML-09 - precision@50 = 0.400 on unseen clients, not the inflated 0.720 I got from an honest looking but leaky random split. That 0.400 is the number I'd actually quote to anyone using this queue.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who this is for:** content reviewers at FlyRank who need to decide which pages to check first, when they don't have time to check every page manually.

**What it's for:** deciding review order, not making the actual title/meta changes automatically. The queue tells you where to look first, a human still reads the page and decides what to change.

**Where it stops being valid:**

- Only tested on pages with impressions_90d >= 500 and avg_position between 1-20, I have no evidence this ranking works for low-traffic or deep-ranked pages, since those were excluded from training entirely.
- Trained and tested only on the clients in this dataset, precision@50 (0.400) was measured on 6 clients the model never saw during training, but if FlyRank onboards a very different kind of client (different industry, very different content style), I have no evidence the model performs the same way there.
- The label itself (needs_review) is a proxy. The top 25% of CTR gaps, not a real human-confirmeC "this page has a title problem." A page can score high here for reasons unrelated to its title (seasonality, cannibalization, a SERP feature stealing the click, the same possibilities I flagged back in ML-07's top-10 review).
- This is decision-support, not a guarantee. A high score means "worth checking first," not "definitely broken."

In [3]:
print(f"impressions range in training data: {visible['impressions_90d'].min():,} to {visible['impressions_90d'].max():,}")
print(f"position range in training data: {visible['avg_position'].min()} to {visible['avg_position'].max()}")
print(f"clients in training data: {visible['client_id'].nunique()}")

impressions range in training data: 500 to 517,715
position range in training data: 0.2 to 20.0
clients in training data: 28


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**What a reviewer must check before acting on a flagged page:**
- Read the actual page and title, not just the score, confirm the title genuinely looks like if it could be improved.
- Check if a sibling page on the same site might be cannibalizing clicks (same concern I raised in ML-07's top-10 review).
- Check if search_volume is 0 or very low for the tracked keyword, if so, the real traffic likely comes from other queries, and a title fix may not help.

**What should never be automated:**
- Automatically rewriting or publishing a new title/meta description without a human reading it first, the model has no understanding of brand voice, accuracy, or whether the new title still matches the page's actual content.
- Automatically deprioritizing or removing pages based on a low score, a low score means "not flagged," not "confirmed fine".
- Using this score as the sole input to any client-facing report or billing decision, it's an internal prioritization tool, not a certified metric.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signs this playbook has gone stale:**
- If the overall base rate (currently 26%, the share of pages with a big CTR gap) shifts a lot over time, that would mean the underlying pattern the model learned no longer matches reality.
- If FlyRank onboards clients very different from the 28 clients in this training data, since I have no evidence the model generalizes beyond what it's seen.
- If a new content_type, provider_used, or model_used value starts showing up that wasn't in training, the model has never seen it and would be guessing.
- If reviewers start reporting that flagged pages "look fine" more often than not. That's a real-world signal the model's precision has dropped below what ML-09 measured (0.400).

**What would trigger a retrain:**
- A meaningful drop in real-world precision reported by reviewers.
- A new quarter/period of data becoming available. Since this model was trained on a single snapshot (March 2026), not something that updates itself.
- A significant shift in the base rate or in feature distributions (e.g., average freshness across all pages changing significantly).

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**Exporting the final queue.** Saving my ranked action queue to work/outputs/ so the capstone report can reference these exact numbers without re-running the whole pipeline.

In [5]:
import os
export_cols = ['content_id', 'client_id', 'model_score', 'action', 'reason_code', 'reason',
               'position_tier', 'avg_position', 'freshness_tier', 'search_volume', 'ctr_gap']

os.makedirs('../outputs', exist_ok=True)
queue[export_cols].to_csv('../outputs/action_playbook_queue.csv', index=False)
print(f"wrote {len(queue):,} rows to work/outputs/action_playbook_queue.csv")

import json
metrics = {
    "honest_precision_at_50": 0.400,
    "base_rate": float(y.mean()),
    "n_scored_pages": len(queue),
    "n_training_clients": int(visible['client_id'].nunique()),
    "impressions_range": [int(visible['impressions_90d'].min()), int(visible['impressions_90d'].max())],
    "position_range": [float(visible['avg_position'].min()), float(visible['avg_position'].max())]
}
with open('../outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("wrote work/outputs/playbook_metrics.json")

wrote 12,023 rows to work/outputs/action_playbook_queue.csv
wrote work/outputs/playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.